# AdaBoost

**Boosting** is a powerful technique that combines many simple models into a *committee* whose performance is far better than any single member. It works best when each individual model is only slightly better than random guessing — such models are called **weak learners**.

The idea is to train the weak learners **in sequence**, each one focusing on the mistakes of its predecessors:
- Each weak learner is trained on **weighted** training data.
- The weights depend on how the previous learners performed: **misclassified points get a higher weight**, so the next learner pays more attention to them.
- The final prediction is a **weighted vote** of all the learners, where more accurate learners get a larger say.

**This notebook covers:**
1. Implementing **AdaBoost** ("Adaptive Boosting") from scratch, with a decision stump as the weak learner.
2. Visualizing how the decision boundary evolves as more learners are added (cf. Figure 14.2 in Bishop, 2006).
3. Applying it to a harder, real data set (handwritten digits) and studying how the training/test error and the quantities $\alpha$ and $\epsilon$ behave as the committee grows.

## Imports

In [ ]:
import matplotlib
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles, load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
matplotlib.rcParams['figure.figsize'] = (4, 3)

## Dataset

We start with a `make_circles` toy set: an inner blob of one class surrounded by a ring of the other. It is **not linearly separable**, so a single straight-line classifier does poorly — a good stress test for boosting. We relabel the two classes to $\{-1, +1\}$, the convention AdaBoost's $\mathrm{sign}$ rule expects.

In [ ]:
np.random.seed(2)
X, y = make_circles(factor=.2, noise=.2)
y[y == 0] = -1  # relabel classes from {0, 1} to {-1, +1}
plt.scatter(X[:, 0], X[:, 1], c=y)
plt.show()

## A Helper for Visualizing Decision Boundaries

The function below draws the classifier's decision regions as a filled contour and overlays the data points. When `sample_weight` is provided, each point's **marker size reflects its current weight** — so heavily weighted (repeatedly misclassified) points appear larger. The weights are clipped first only so the markers stay within a readable size range.

In [ ]:
def plot_contour(clf, X, y, sample_weight=None):
    xx, yy = np.mgrid[-3:3:.1, -3:3:.1]
    zz = np.c_[xx.ravel(), yy.ravel()]
    proba = clf.predict(zz).reshape(xx.shape)

    # Use the sample weights as marker sizes (clipped to a readable range).
    if sample_weight is not None:
        sample_weight = np.clip(sample_weight, 20, 500)

    plt.scatter(X[:, 0], X[:, 1], c=y, zorder=2, s=sample_weight, cmap='coolwarm', edgecolors='k')
    plt.contourf(xx, yy, proba, zorder=1, alpha=.5, cmap='coolwarm')
    plt.show()

## The AdaBoost Algorithm

AdaBoost builds the committee one weak learner at a time. Every data point $n$ carries a weight $w_n$ that tells the next learner how much that point matters. Starting from uniform weights $w_n = 1/N$, we repeat for $m = 1, \dots, M$:

1. **Train** a weak learner $y_m(\mathbf{x})$ on the data weighted by $w_n$ (here a *decision stump* — a depth-1 tree).
2. **Weighted error:** the fraction of weight sitting on the misclassified points,
$$\epsilon_m = \frac{\sum_{n:\,y_m(\mathbf{x}_n)\neq t_n} w_n}{\sum_n w_n}.$$
3. **Model weight:** give the learner a vote based on how good it is,
$$\alpha_m = \ln\frac{1-\epsilon_m}{\epsilon_m}.$$
A learner barely better than chance ($\epsilon_m \to 0.5$) gets $\alpha_m \to 0$; a very accurate one gets a large $\alpha_m$.

4. **Reweight:** multiply the weight of every misclassified point by $e^{\alpha_m}$ (leaving correct ones unchanged), so the next learner concentrates on the hard cases.

The final prediction is the **weighted vote** of all learners:
$$Y(\mathbf{x}) = \mathrm{sign}\!\left(\sum_{m=1}^M \alpha_m\, y_m(\mathbf{x})\right).$$

> **Implement AdaBoost** in the `Boosting` class below, using `DecisionTreeClassifier(max_depth=1)` as the weak learner. Its `.fit(X, y, sample_weight=...)` argument passes the per-sample weights into the training loss.

**Python note — decision stump:** `max_depth=1` builds a tree that splits on a single feature exactly once — a deliberately *weak* learner. Boosting's power comes from combining many such simple rules, not from any single one.

**Python note — growing weights:** we only ever *increase* misclassified weights (multiply by $e^{\alpha_m} > 1$) and never renormalize them. That is fine because $\epsilon_m$ divides by $\sum_n w_n$, so the overall scale cancels out. The weights can grow large over the iterations — which is exactly why `plot_contour` clips them before using them as marker sizes.

In [ ]:
class Boosting:
    def __init__(self, n_models=100):
        self.n_models = n_models

    def fit(self, X, y, plot=False):
####################
# Your Code Here   #
####################

    def predict(self, X):
####################
# Your Code Here   #
####################


clf = Boosting(n_models=20)
clf.fit(X, y, plot=False)
plot_contour(clf, X, y, clf.sample_weight)

> **Visualize how the decision boundary evolves** as more weak learners are added (compare with Figure 14.2 in Bishop, 2006).

**What to look for:**
- The earliest boundaries are simple straight cuts — that is all a single stump can produce.
- As learners accumulate, the combined boundary bends around the ring and separates the classes.
- Misclassified points grow in size from iteration to iteration as their weights increase, showing the committee shifting its focus onto them.

In [ ]:
####################
# Your Code Here   #
####################

## A Harder Dataset: Digits 1 vs. 7

Now we test on real data: the handwritten digits from `load_digits`, restricted to the visually similar digits **1 and 7**. We map digit $1 \to +1$ and digit $7 \to -1$, then split into train and test sets to measure generalization. Each digit is an $8 \times 8$ grayscale image flattened into a 64-dimensional vector.

In [ ]:
X, y = load_digits(return_X_y=True)
idx = (y == 1) | (y == 7)
X, y = X[idx], y[idx]
y[y == 1] = 1
y[y == 7] = -1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

plt.imshow(X_train[np.random.randint(len(X_train))].reshape(8, 8))
plt.show()

> **Train Boosting models with an increasing number of estimators.** For each model, store the training error, test error, and the average $\alpha$ and $\epsilon$ across its weak learners, so we can study how these quantities behave as the committee grows.

In [ ]:
####################
# Your Code Here   #
####################

> **Plot the training and test error against the number of estimators.**

**What to look for:** the **training error** drops quickly and usually reaches zero, while the **test error** levels off at a low value. AdaBoost keeps improving the classification margin even after the training error hits zero, which makes it fairly resistant to overfitting on this task.

In [ ]:
####################
# Your Code Here   #
####################

> **Inspect the average weighted error $\epsilon$ and model weight $\alpha$** across the weak learners.

**What to look for:** as the committee grows, each new stump faces a harder, reweighted problem, so the average $\epsilon$ tends to **rise toward $0.5$** (the chance level) and the average $\alpha$ correspondingly **falls toward $0$** — later learners contribute smaller, more specialized corrections.

In [ ]:
####################
# Your Code Here   #
####################

In [ ]:
####################
# Your Code Here   #
####################

#### Discussion questions

1. What happens to the model weight $\alpha_m$ when a weak learner is no better than random guessing ($\epsilon_m = 0.5$)?
####################
 Your Text Here   
####################

2. Why do we use *weak* learners (decision stumps) instead of a single strong learner?
####################
 Your Text Here   
####################

3. Why does increasing the weight of misclassified points help?
####################
 Your Text Here   
####################